In [1]:
import torch
import numpy as np
import import_ipynb
%load_ext autoreload
%autoreload 2
from a02_functions import ClimbCNN


# Task 1: Climbing with a CNN

Initialize `model` using the right hyperparameters for task 1a). Then inspect your
model's parameters using `model.state_dict()`. (Note that when using layer
implementations from `torch.nn`, parameters are initialized randomly as discussed in
the lecture.)

In [2]:
# TODO: your code here
# in_channel is 1 because input is one single elevation value
# out_channel is 1 since i only need ascent
# => 1d cnn
# kernel size is 2 because it measures ascents and descents
model = ClimbCNN(in_channels=1, out_channels=1, kernel_size=2)

You can access the model parameters via `<model>.<param-name>`. Set all parameters to
the values required to solve task a).

In [3]:
with torch.no_grad():  # needed so that you can assign values to the model parameters
    # TODO: your code here
    # i define filter weights and biass
    print(model.state_dict())
    model.conv.weight[:] = torch.tensor([[[-1.0, 1.0]]])
    model.conv.bias[:] = 0.0
    print(model.state_dict())

OrderedDict({'conv.weight': tensor([[[-0.0637, -0.2461]]]), 'conv.bias': tensor([-0.3321])})
OrderedDict({'conv.weight': tensor([[[-1.,  1.]]]), 'conv.bias': tensor([0.])})


Simple test case that can be verified by hand.

In [4]:
x = torch.Tensor([0.0, 5.0, 11.0, 7.0, 15.0, 3.0]).view(1, -1)
# by hand i calculate: 
# window 1 = -1*0 + 1*5 = 5
# window 2 = -1*5 + 1*11 = 6
# window 3 = -1*11 + 1*7 = -4
# window 4 = -1*7 + 1*15 = 8
# window 5 = -1*15 + 1*3 = -12
# feature map is then = [5,6,-4,8,-12]
#  AF = ReLu = max(0, x) = [5,6,0,8,0]
# sum => 19 => correct
with torch.no_grad():
    y = model(x)
print(y)  # should give: tensor([19.])

tensor([19.])


Example from the lecture.

In [5]:
climb_data = (
    torch.from_numpy(np.genfromtxt("climb_filtered.csv")).to(torch.float).view(1, -1)
)
with torch.no_grad():
    y = model(torch.Tensor(climb_data).view(1, -1))

print(y)  # should give: tensor([503.4000])

tensor([503.4000])


Now create a new `model2` that computes that total ascent (first output) and total descent (second output) simultaneously. Do this using the same model implementation as above, but change only hyperparameters and parameters.

In [6]:
# TODO: your code here
# same input as before, but now i want an additional value the descent
model2 = ClimbCNN(in_channels=1, out_channels=2, kernel_size=2)

with torch.no_grad():  # needed so that you can assign values to the model parameters
    # TODO: your code here
    # i define filter weights and biass
    # need an additional filter for the second output
    print(model2.state_dict())
    model2.conv.weight[:] = torch.tensor([[[-1.0, 1.0]], [[1.0, -1.0]]])
    model2.conv.bias[:] = 0.0
    print(model2.state_dict())

OrderedDict({'conv.weight': tensor([[[-0.5902,  0.4761]],

        [[-0.0915,  0.4405]]]), 'conv.bias': tensor([ 0.6705, -0.4438])})
OrderedDict({'conv.weight': tensor([[[-1.,  1.]],

        [[ 1., -1.]]]), 'conv.bias': tensor([0., 0.])})


In [7]:
x = torch.Tensor([0.0, 5.0, 11.0, 7.0, 15.0, 3.0]).view(1, -1)
with torch.no_grad():
    y = model2(x)
    # by hand i calculate: 
    # window 1 = -1*0 + 1*5 = 5, 1*0 + -1*5 = -5
    # window 2 = -1*5 + 1*11 = 6, 1 *5 + -1*11 = -6
    # window 3 = -1*11 + 1*7 = -4, 1*11 + -1*7 = 4
    # window 4 = -1*7 + 1*15 = 8, 1*7 + -1*15 = -8
    # window 5 = -1*15 + 1*3 = -12, 1*15 + -1*3 = 12
    # feature map is then = [[5,6,-4,8,-12], [-5, -6, 4, -8, 12]]
    #  AF = ReLu = max(0, x) = [[5,6,0,8,0], [0, 0, 4, 0, 12]]
    # sum => [19, 16] => correct
print(y)  # should give: [19., 16.]

tensor([19., 16.])


In [8]:
with torch.no_grad():
    y = model2(torch.Tensor(climb_data).view(1, -1))

print(y)  # should give: tensor([503.4000, 513.6000])

tensor([503.4000, 513.6000])
